In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from romlab import FOM, AdaptiveSampling, docker_solver

DATA = (ROOT / "data/2d/lambda-adaptive").resolve()
FIELD = "velocity"      # "velocity" or "c-trace"
ACQ = "alc"            # "var", "alc" or "lhs"
SEED = 1
MAX_SAMPLES = 25     # 5 initial runs + 15 adaptive iterations (~40 s per full-order solve)

fom = FOM(
    data_folder=DATA,
    filenames_train=["snapshots.txt", "c-trace.txt", "mesh_coor2.txt", "parameters.txt"],
    filenames_test=["snapshots_test.txt", "c-trace_test.txt", "mesh_coor2_test.txt", "parameters_test.txt"],
    param_cols=[0], seed=42)   
fom.info()
initial = fom.parameters_train[:, 0].copy()

In [ ]:
solver = docker_solver(str(DATA), template_row=fom.parameters_train[0], param_cols=fom.param_cols)
adaptive = AdaptiveSampling(fom, solver, eps=1e-6, field=FIELD, acq=ACQ, n_candidates=10_000, plots=True, seed=SEED)
history = adaptive.run(max_samples=MAX_SAMPLES)

In [ ]:
tag = f"{FIELD}_{ACQ}_{SEED}"
results = DATA / "results"
results.mkdir(exist_ok=True)

rows = [[h["n_train"], h["nmodes"], h["error"], h["max_std"], *h.get("mu", [np.nan]), h.get("std", np.nan)]
        for h in history]
cols = [("M", 5, "d"), ("r", 5, "d"), ("mean_error", 14, ".6e"), ("max_std", 14, ".6e"),
        ("next_lambda", 13, ".6f"), ("next_std", 14, ".6e")]
names = " ".join(f"{n:>{w}}" for n, w, _ in cols)  # header names right-aligned over their columns
np.savetxt(results / f"{tag}_results.txt", rows, fmt=[f"%{w}{s}" for _, w, s in cols], comments="",
           header=f"# field = {FIELD}, acq = {ACQ}, seed = {SEED}\n#{names[1:]}")

In [ ]:
added = fom.parameters_train[len(initial):, 0]
M = [h["n_train"] for h in history]

plt.rcParams.update({"font.size": 14, "axes.labelsize": 16, "xtick.labelsize": 13, "ytick.labelsize": 13})
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]  # where the points went: lambda of every sample against the iteration that added it
ax.plot(fom.parameters_test[:, 0], np.full(len(fom.parameters_test), -1), "|", c="lightgray", ms=12, label="test")
ax.scatter(initial, np.zeros(len(initial)), c="k", marker="s", label="initial")
sc = ax.scatter(added, np.arange(1, len(added) + 1), c=np.arange(1, len(added) + 1), cmap="viridis",
                label=f"added ({ACQ})")
ax.set(xlabel=r"$\lambda$", ylabel="iteration")
ax.legend(loc="upper right", fontsize=10)

axes[1].semilogy(M, [h["error"] for h in history], "o-")
axes[1].set(xlabel="training samples $M$", ylabel="mean relative error", title=f"{FIELD}, acq = {ACQ}")
axes[2].semilogy(M, [h["max_std"] for h in history], "o-")
axes[2].set(xlabel="training samples $M$", ylabel="max predictive std")
plt.tight_layout()
plt.show()